# 03. ASR Evaluation

## Goal
Calculate Character Error Rate (CER) and Word Error Rate (WER) for the Whisper baseline on a small LOTUSDIS sample.

## Why This Step Matters
Manual inspection tells us what the model gets wrong, but CER and WER give us numbers we can compare against future fine-tuning runs.

## Inputs
- Streamed `nectec/LOTUSDIS` samples from Hugging Face.
- `openai/whisper-tiny` baseline predictions.
- Normalized reference and prediction text.

## Outputs
- Overall CER and WER.
- Best and worst examples by CER.
- A small list of visible error patterns.


## Setup

Import the baseline ASR tools, text normalization, and metric functions.

In [6]:
from collections import Counter
from pathlib import Path
import sys

import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline

sys.path.append(str(Path("..").resolve()))
from src.asr.metrics import character_error_rate, word_error_rate
from src.asr.normalize_text import normalize_text

DEBUG = True
MAX_SAMPLES = 5 if DEBUG else 50
MODEL_NAME = "openai/whisper-tiny"

print("CUDA Available:", torch.cuda.is_available())


CUDA Available: True


## Load Data and Model

Use streaming mode so we do not download the full dataset into the repository.

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {device}...")
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=device,
)

dataset = load_dataset("nectec/LOTUSDIS", streaming=True)
samples = []
iterator = iter(dataset["train"])

for _ in range(MAX_SAMPLES):
    samples.append(next(iterator))

print(f"Loaded {len(samples)} samples.")


Loading openai/whisper-tiny on cuda...


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 12353.81it/s]


Loaded 5 samples.


## Run Evaluation

Normalize references and predictions before scoring. This keeps punctuation and spacing differences from dominating the metric.

In [8]:
rows = []

for index, sample in enumerate(samples, start=1):
    print(f"Evaluating sample {index}/{len(samples)}...")
    reference = sample["sentence"]
    prediction = asr_pipe(
        sample["audio"],
        generate_kwargs={"language": "thai", "task": "transcribe"},
    )["text"]

    norm_reference = normalize_text(reference)
    norm_prediction = normalize_text(prediction)

    rows.append(
        {
            "sample": index,
            "reference": reference,
            "prediction": prediction,
            "norm_reference": norm_reference,
            "norm_prediction": norm_prediction,
            "cer": character_error_rate(norm_reference, norm_prediction),
            "wer": word_error_rate(norm_reference, norm_prediction),
        }
    )

results_df = pd.DataFrame(rows)
results_df[["sample", "cer", "wer", "norm_reference", "norm_prediction"]]


Evaluating sample 1/5...
Evaluating sample 2/5...
Evaluating sample 3/5...
Evaluating sample 4/5...
Evaluating sample 5/5...


,sample,cer,wer,norm_reference,norm_prediction
0,1,0.575758,1.000000,ครับสวัสดีครับ m ศูนย์หนึ่งครับผม,ครับ ส่วนรึงครับผม
1,2,0.400000,0.666667,สวัสดีค่ะ f ศูนย์หนึ่งค่ะ,สวัสดีค่ะ แฟฟสุดลึงค่ะ
2,3,0.260870,0.666667,สวัสดีค่ะ f ศูนย์สองค่ะ,สวัสดีค่ะ ฝฟุ้นสองค่ะ
3,4,0.774194,4.666667,หัวข้อ topic ที่ห้าสิบเจ็ดชอบกินข้าวนอกบ้านหรื...,หวักค่ะ ทอบปิกที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ...
4,5,0.333333,1.000000,ครับก็,ครับ


## Inspect Results

Average scores give the headline result. Best and worst examples show what the number actually means.

In [9]:
overall_cer = results_df["cer"].mean()
overall_wer = results_df["wer"].mean()

print(f"Overall CER: {overall_cer:.2%}")
print(f"Overall WER: {overall_wer:.2%}")

display(results_df.sort_values("cer")[["sample", "cer", "wer", "reference", "prediction"]])


Overall CER: 46.88%
Overall WER: 160.00%


,sample,cer,wer,reference,prediction
2,3,0.260870,0.666667,สวัสดีค่ะ F ศูนย์สองค่ะ,สวัสดีค่ะ ฝฟุ้นสองค่ะ
4,5,0.333333,1.000000,ครับก็,ครับ
1,2,0.400000,0.666667,สวัสดีค่ะ F ศูนย์หนึ่งค่ะ,สวัสดีค่ะ แฟฟสุดลึงค่ะ
0,1,0.575758,1.000000,ครับสวัสดีครับ M ศูนย์หนึ่งครับผม,ครับ ส่วนรึงครับผม
3,4,0.774194,4.666667,หัวข้อ Topic ที่ห้าสิบเจ็ดชอบกินข้าวนอกบ้านหรื...,หวักค่ะ ทอบปิกที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ...


## Common Error Patterns

These are simple checks for visible baseline problems. They are not a replacement for manual listening.

In [10]:
patterns = Counter()

for row in rows:
    reference = row["norm_reference"]
    prediction = row["norm_prediction"]

    if len(prediction) < len(reference):
        patterns["shorter prediction / possible omission"] += 1
    if len(prediction) > len(reference):
        patterns["longer prediction / possible insertion"] += 1
    if row["wer"] >= 1.0:
        patterns["most words changed"] += 1

patterns


Counter({'shorter prediction / possible omission': 4,
         'most words changed': 3,
         'longer prediction / possible insertion': 1})

## What We Learned

- CER is useful for Thai because word boundaries are not always reliable.
- WER is still useful for quick inspection when the transcript contains spaces.
- Whisper-tiny gives us a baseline, but the early examples show substitutions, omissions, and repeated words.

## Common Problems

- If `torch` or `transformers` is missing, install the project requirements in the selected notebook kernel.
- If Hugging Face streaming is slow, rerun the cell or configure a Hugging Face token outside the notebook.
- If CUDA is unavailable, the notebook still works on CPU but will run slower.

## Next Step

Use these metrics as the baseline score before testing larger models or fine-tuning.
